# 01 · Data Acquisition — Real Macro Drivers
**Principle: real public / reference data first, synthetic business data second.**

This notebook inspects the *real* external data the project is built on: commodity, energy and FX series pulled from **FRED** (St. Louis Fed), plus a curated country reference table. Produced by `python -m src.data_acquisition`.

In [ ]:
%matplotlib inline
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40, "display.width", 160)
from src import config as C
print("project root:", ROOT)

## Provenance
Every run records where each dataset came from (live FRED vs documented fallback).

In [ ]:
import json
meta = json.loads(C.ACQUISITION_META.read_text())
print('window:', meta['window_start'], '->', meta['window_end'])
for d in meta['datasets']:
    print(f"  {d['dataset']:<26} source={d['source']:<24} rows={d['rows']}")

## The three real index series

In [ ]:
com = pd.read_csv(C.COMMODITY_CSV, parse_dates=['date'])
ene = pd.read_csv(C.ENERGY_CSV, parse_dates=['date'])
fx  = pd.read_csv(C.EXCHANGE_CSV, parse_dates=['date'])
com.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11,5.5))
ax.plot(com.date, com.raw_material_index, lw=3, color='#b3122b', label='Raw-material index')
ax.plot(ene.date, ene.energy_cost_index, lw=2, ls='--', color='#1f77b4', label='Energy cost index')
ax.plot(fx.date,  fx.exchange_rate_index, lw=2, ls=':', color='#2ca02c', label='EUR/USD index')
ax.axhline(100, color='k', lw=.8, alpha=.4)
ax.set_title('Real macro drivers (FRED), rebased to 100 at Jan-2022'); ax.legend(); plt.show()

**Read-out.** The data carries genuine economic history: the **2022 energy crisis** (energy index peaking ~156, +56%), cooling commodity markets afterwards, and a **weakening euro**. These real movements drive the synthetic prices in notebook 02.

In [ ]:
country = pd.read_csv(C.COUNTRY_CSV)
country